<div style="padding:28px;border-radius:18px;background:linear-gradient(135deg,#0f172a,#1e3a5f);color:white">
  <div style="font-size:13px;letter-spacing:2px;font-weight:700">TOPIC 766 · TUTORIAL OF AGENTIC SYSTEMS</div>
  <h1 style="margin:8px 0 6px 0;font-size:34px">Notebook 2 — Build Constellations</h1>
  <div style="font-size:18px;opacity:.9">Agents → Roles → Collaboration · From one M&A generalist to a coordinated specialist team</div>
</div>

### Introduction

Notebook 1 showed that a single agent can act. It could interpret an M&A mandate, choose deterministic tools, inspect structured and unstructured evidence, and produce an acquisition recommendation. That was an important milestone, but it also revealed a limitation: one agent had to behave simultaneously like a financial analyst, a strategic adviser, an intelligence researcher, and a senior deal lead. The system had capabilities, but it did not yet have organization.

Notebook 2 introduces the second step of the pedagogical ladder: **constellations**. A constellation is a coordinated group of specialized agents that work on different parts of the same problem and then combine their outputs. Our M&A setting makes the motivation intuitive. A financial specialist should focus on valuation, growth, leverage, and affordability. A strategy specialist should examine sector fit, geographic complementarity, business models, strengths, weaknesses, and integration difficulty. An intelligence specialist should inspect financial-report excerpts, analyst notes, and rumors while distinguishing reliable evidence from uncertain chatter. A deal-lead agent should then reconcile those perspectives and formulate the final recommendation.

The objective is not to prove that four agents are always better than one. The objective is to understand what changes when cognition is distributed across roles. Specialization can improve focus, make responsibilities clearer, and create more inspectable intermediate outputs. But it also creates a new systems problem: collaboration. Specialists may disagree, duplicate work, use different assumptions, or produce outputs that are difficult to combine.

For that reason, this notebook will build the constellation progressively. We will reuse the same synthetic 500-company M&A universe, create role-specific tool access, define explicit specialist contracts, execute independent analyses, and then introduce a deal-lead agent that synthesizes them. By the end, the learner should understand not merely how to run several agents, but how **roles, boundaries, shared context, and message contracts transform multiple agents into one coordinated system**.

<div style="padding:20px;border-left:6px solid #2563eb;background:#eff6ff;border-radius:10px">
<b>What we need to understand and learn</b>
</div>

The first concept in this notebook is **specialization**. In Notebook 1, the agent had access to several tools and one M&A assessment skill. It could move between financial analysis, strategy, document search, and final synthesis. That architecture is compact, but every kind of reasoning competes inside one role. A constellation separates these responsibilities. The purpose is not merely to produce more text. Each specialist should have a narrower objective, a smaller relevant action space, and an explicit responsibility for part of the final answer.

The second concept is the **role**. A role is more than a title in a prompt. It defines what an agent is responsible for, which evidence it may use, which tools it may call, and what output it must deliver. A financial specialist that is allowed to search rumors and make final strategic recommendations is poorly separated from the rest of the team. A useful role boundary reduces ambiguity. In this notebook, the Financial Analyst receives tools for company retrieval, candidate screening, and financial comparison. The Strategy Analyst receives tools for strategic comparison and company profiles. The Intelligence Analyst receives tools for document retrieval and source assessment. The Deal Lead does not repeat every specialist's work; it receives their outputs and owns synthesis.

The third concept is **collaboration through artifacts**. Agents do not need to share hidden reasoning to collaborate. They need useful intermediate products. We will therefore make each specialist return a compact structured memorandum with a recommendation, supporting evidence, risks, and uncertainty. These outputs become the shared artifacts of the constellation. The Deal Lead can inspect them, identify agreement or disagreement, and decide how to reconcile the evidence.

The fourth concept is **coordination cost**. Once we distribute cognition, we gain specialization but lose the simplicity of one mind handling the whole problem. Agents can choose different candidates, use different thresholds, or interpret the same company differently. This is not a failure of the exercise. It is precisely the phenomenon we need to expose. A constellation requires contracts: common company IDs, explicit output schemas, a shared mission, and a defined authority for final synthesis.

The fifth concept is **orchestration**. In this notebook, orchestration remains deliberately simple and mostly predetermined. We know the roles in advance: Financial Analyst, Strategy Analyst, Intelligence Analyst, and Deal Lead. We also know the broad sequence: specialists analyze, then the lead synthesizes. This makes the architecture easy to inspect. Later, in Notebook 4, the system itself will decide which roles are needed. For now, the organization is designed by us.

A sixth concept is **information locality**. A specialist should receive the information it needs without automatically inheriting everything known by the entire system. Narrower context can make responsibilities clearer and reduce accidental duplication. In our constellation, specialists share stable company IDs and a common mission, but they interact with different evidence channels. This is a first, simple form of capability scoping.

Finally, we need to distinguish a constellation from a mere batch of independent prompts. The constellation becomes a system only when the agents work on a common mission, operate under complementary responsibilities, exchange standardized outputs, and contribute to a combined decision. That distinction is the central learning objective of Notebook 2.

## A Note about Skills

In this Notebook, the concept of 'skills' is primarily embedded in the `purpose` and `allowed_tools` defined for each agent's `role`.

Let me explain:

*   **Role Purpose**: Each agent (Financial Analyst, Strategy Analyst, Intelligence Analyst, Deal Lead) has a clearly defined `purpose` (as shown in `ROLE_CONFIG` in Code Unit 3).
    *   For example, the Financial Analyst's purpose is to "Evaluate M&A candidates using growth, profitability, valuation, leverage, and financial capacity." This purpose implicitly defines its core 'skills' or area of expertise.

*   **Allowed Tools**: Crucially, each role is given a *restricted* set of `allowed_tools`. This is how the 'skill' or capability of an agent is architecturally enforced.
    *   For instance, the Financial Analyst has tools like `get_company_snapshot`, `screen_candidates`, and `compare_financials`, but it *cannot* use `search_documents`. This explicitly limits its operational 'skills' to financial analysis.
    *   The Intelligence Analyst, on the other hand, *can* `search_documents` but cannot directly perform financial comparisons.

So, while 'skill' isn't an explicit, separate parameter in the agent definition, it's a direct outcome of the **role's purpose** and the **tools it's permitted to use**.

The notebook emphasizes moving from one generalist agent with many skills (Notebook 1) to specialized agents, each with a narrower purpose and a limited set of tools, which collectively define their 'skill set' for the constellation.

### Code Unit 1 of 10 — Reconnect to the common M&A world

The first code unit restores the shared environment and keeps the course architecture consistent. We mount Google Drive, point to the common dataset folder, retrieve `OPENAI_API_KEY` from Colab Secrets, initialize the OpenAI client, and load the four student-facing files generated in Notebook 0. As before, the teacher-only benchmark is not loaded. The structured tables are joined into one analytical universe, while the document corpus remains separate because textual intelligence will be assigned to a specialist role later. This cell also performs a compact integrity check so that the constellation is not built on corrupted inputs. Pedagogically, the repetition is intentional. Students should recognize that the environment has not changed since Notebook 1. The new learning comes from the organization of agency, not from quietly changing the data. Holding the M&A world fixed allows us to attribute differences in behavior to specialization and collaboration rather than to a different problem or richer information.

In [1]:
%pip -q install -U openai

from google.colab import drive, userdata
from openai import OpenAI
from pathlib import Path
import pandas as pd
import numpy as np
import json, re
from typing import Any

drive.mount("/content/drive")

DATASET_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "TOPIC_766 TUTORIAL OF AGENTIC SYSTEMS/DATASET"
)

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Add OPENAI_API_KEY in Colab → Secrets before continuing.")

MODEL = "gpt-5.2"
client = OpenAI(api_key=OPENAI_API_KEY)

companies = pd.read_csv(DATASET_DIR / "companies.csv")
financials = pd.read_csv(DATASET_DIR / "financials.csv")
strategic_profiles = pd.read_csv(DATASET_DIR / "strategic_profiles.csv")
documents = pd.read_csv(DATASET_DIR / "documents.csv")

universe = (
    companies
    .merge(financials, on="company_id", validate="one_to_one")
    .merge(strategic_profiles, on="company_id", validate="one_to_one")
)

assert len(universe) == 500
assert universe["company_id"].is_unique
assert documents.groupby("company_id").size().eq(3).all()

print("✓ Shared M&A world loaded.")
print("Companies:", len(universe))
print("Documents:", len(documents))
print("Model:", MODEL)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 13.7 MB/s eta 0:00:00
Mounted at /content/drive
✓ Shared M&A world loaded.
Companies: 500
Documents: 1500
Model: gpt-5.2


### Code Unit 2 of 10 — Rebuild the deterministic tool layer as shared infrastructure

A constellation should not require every specialist to invent its own version of the data. This cell creates a compact shared tool layer that all roles can draw from according to their permissions. We define five deterministic operations: retrieve a company snapshot, screen candidate targets, compare financial characteristics, compare strategic characteristics, and search unstructured documents. The tools are intentionally similar to those introduced in Notebook 1, because the conceptual object has not changed. What changes in Notebook 2 is **who is allowed to use which tool and for what purpose**. This shared infrastructure gives the constellation a common factual substrate. A financial specialist and a strategy specialist may interpret a company differently, but they refer to the same company ID and the same underlying data. That prevents disagreement from being caused by inconsistent databases. The cell therefore illustrates a foundational architectural principle: specialization should occur above a stable layer of shared facts and deterministic capabilities whenever possible.

In [2]:
def get_company_snapshot(company_id: str) -> dict:
    match = universe[universe["company_id"] == company_id]
    if match.empty:
        return {"ok": False, "error": f"Unknown company_id: {company_id}"}
    return {"ok": True, "company": json.loads(match.to_json(orient="records"))[0]}


def screen_candidates(
    exclude_company_id: str,
    sector: str,
    continent: str,
    min_growth_pct: float,
    max_ev_ebitda: float,
    max_net_debt_ebitda: float,
    limit: int,
) -> dict:
    df = universe.copy()
    if exclude_company_id != "NONE":
        df = df[df["company_id"] != exclude_company_id]
    if sector != "ANY":
        df = df[df["sector"] == sector]
    if continent != "ANY":
        df = df[df["continent"] == continent]

    df = df[
        (df["revenue_growth_pct"] >= min_growth_pct)
        & (df["ev_ebitda"] <= max_ev_ebitda)
        & (df["net_debt_ebitda"] <= max_net_debt_ebitda)
    ].copy()

    if df.empty:
        return {"ok": True, "count": 0, "candidates": []}

    df["screen_score"] = (
        0.50 * df["revenue_growth_pct"]
        - 0.30 * df["ev_ebitda"]
        - 0.20 * df["net_debt_ebitda"]
    )
    cols = [
        "company_id", "company_name", "country", "continent", "sector", "subsector",
        "revenue_growth_pct", "ebitda_margin_pct", "ev_ebitda",
        "net_debt_ebitda", "screen_score",
    ]
    result = df.sort_values("screen_score", ascending=False).head(int(limit))[cols]
    return {
        "ok": True,
        "count": len(result),
        "candidates": json.loads(result.to_json(orient="records")),
    }


def compare_financials(company_ids: list[str]) -> dict:
    cols = [
        "company_id", "company_name", "revenue_usd_m", "ebitda_usd_m",
        "ebitda_margin_pct", "revenue_growth_pct", "enterprise_value_usd_m",
        "ev_ebitda", "net_debt_ebitda"
    ]
    df = universe[universe["company_id"].isin(company_ids)][cols].copy()
    if len(df) < 2:
        return {"ok": False, "error": "Provide at least two valid company IDs."}

    df["financial_score"] = (
        0.40 * np.clip((df["revenue_growth_pct"] + 10) / 40, 0, 1)
        + 0.35 * np.clip((26 - df["ev_ebitda"]) / 21, 0, 1)
        + 0.25 * np.clip((6 - df["net_debt_ebitda"]) / 6, 0, 1)
    )
    df = df.sort_values("financial_score", ascending=False)
    return {"ok": True, "comparison": json.loads(df.to_json(orient="records"))}


def compare_strategy(buyer_id: str, target_ids: list[str]) -> dict:
    buyer = universe[universe["company_id"] == buyer_id]
    targets = universe[universe["company_id"].isin(target_ids)].copy()
    if buyer.empty or len(targets) < 2:
        return {"ok": False, "error": "Buyer must exist and at least two targets are required."}

    b = buyer.iloc[0]
    targets["same_sector"] = targets["sector"].eq(b["sector"])
    targets["same_continent"] = targets["continent"].eq(b["continent"])
    targets["cross_border"] = targets["country"].ne(b["country"])
    targets["strategy_score"] = (
        0.35 * targets["same_sector"].astype(float)
        + 0.15 * targets["same_continent"].astype(float)
        + 0.20 * targets["cross_border"].astype(float)
        + 0.15 * targets["integration_complexity"].map({"Low": 1.0, "Moderate": 0.6, "High": 0.2})
        + 0.15 * targets["regulatory_sensitivity"].map({"Low": 1.0, "Moderate": 0.6, "High": 0.2})
    )

    cols = [
        "company_id", "company_name", "country", "sector", "subsector",
        "business_model", "strategic_strength", "strategic_weakness",
        "same_sector", "same_continent", "cross_border",
        "integration_complexity", "regulatory_sensitivity", "strategy_score",
    ]
    return {
        "ok": True,
        "buyer_id": buyer_id,
        "comparison": json.loads(
            targets.sort_values("strategy_score", ascending=False)[cols]
            .to_json(orient="records")
        ),
    }


def search_documents(query: str, company_id: str, document_type: str, limit: int) -> dict:
    df = documents.copy()
    if company_id != "ANY":
        df = df[df["company_id"] == company_id]
    if document_type != "ANY":
        df = df[df["document_type"] == document_type]

    query_tokens = [t for t in re.findall(r"[a-z0-9]+", query.lower()) if len(t) > 2]
    df["search_score"] = df["text"].str.lower().apply(
        lambda x: sum(x.count(t) for t in query_tokens)
    )
    df = df[df["search_score"] > 0].sort_values("search_score", ascending=False)

    cols = [
        "document_id", "company_id", "document_type",
        "source_reliability", "search_score", "text"
    ]
    result = df.head(int(limit))[cols]
    return {
        "ok": True,
        "count": len(result),
        "documents": json.loads(result.to_json(orient="records")),
    }

print("Shared deterministic tools are ready.")

Shared deterministic tools are ready.


### Code Unit 3 of 10 — Define specialist roles, tool permissions, and output contracts

This is the architectural heart of the notebook. We now define the constellation not as “four agents” but as four **roles with boundaries**. Each specialist receives a purpose, a restricted set of tools, and a required output contract. The Financial Analyst can retrieve companies, screen targets, and compare financials. The Strategy Analyst can retrieve companies and compare strategic fit. The Intelligence Analyst can retrieve companies and search documents. The Deal Lead is different: it receives the specialists’ memoranda and owns the integrated recommendation. Restricting tools is pedagogically valuable because it makes specialization observable. If every agent can do everything, role names become decorative rather than architectural. We also define a common specialist memorandum format containing a preferred candidate, alternatives, evidence, risks, and uncertainty. This standardized artifact is what enables later collaboration. In a multi-agent system, reliable coordination often depends less on eloquent prompts than on disciplined interfaces between components.

In [3]:
ROLE_CONFIG = {
    "financial_analyst": {
        "title": "Financial Analyst",
        "purpose": (
            "Evaluate M&A candidates using growth, profitability, valuation, leverage, "
            "and financial capacity. Do not make claims from rumors or strategic narratives."
        ),
        "allowed_tools": ["get_company_snapshot", "screen_candidates", "compare_financials"],
    },
    "strategy_analyst": {
        "title": "Strategy Analyst",
        "purpose": (
            "Evaluate strategic fit, sector and geographic complementarity, business models, "
            "strengths, weaknesses, integration complexity, and regulatory sensitivity."
        ),
        "allowed_tools": ["get_company_snapshot", "compare_strategy"],
    },
    "intelligence_analyst": {
        "title": "Intelligence Analyst",
        "purpose": (
            "Inspect unstructured evidence, distinguish source reliability, identify rumors, "
            "and surface information that may confirm or challenge the structured case."
        ),
        "allowed_tools": ["get_company_snapshot", "search_documents"],
    },
    "deal_lead": {
        "title": "Deal Lead",
        "purpose": (
            "Integrate specialist memoranda, reconcile disagreement, distinguish facts from "
            "judgment, and produce the final M&A recommendation."
        ),
        "allowed_tools": [],
    },
}

SPECIALIST_OUTPUT_CONTRACT = {
    "preferred_candidate": "Company ID or NONE",
    "alternatives": ["Company ID", "..."],
    "key_evidence": ["concise evidence item", "..."],
    "main_risks": ["risk item", "..."],
    "uncertainties": ["uncertainty item", "..."],
    "specialist_conclusion": "short conclusion",
}

print("CONSTELLATION ROLES")
for key, cfg in ROLE_CONFIG.items():
    print(f"\n{cfg['title']}")
    print(" Purpose:", cfg["purpose"])
    print(" Tools:", cfg["allowed_tools"])

print("\nShared specialist output contract:")
print(json.dumps(SPECIALIST_OUTPUT_CONTRACT, indent=2))

CONSTELLATION ROLES

Financial Analyst
 Purpose: Evaluate M&A candidates using growth, profitability, valuation, leverage, and financial capacity. Do not make claims from rumors or strategic narratives.
 Tools: ['get_company_snapshot', 'screen_candidates', 'compare_financials']

Strategy Analyst
 Purpose: Evaluate strategic fit, sector and geographic complementarity, business models, strengths, weaknesses, integration complexity, and regulatory sensitivity.
 Tools: ['get_company_snapshot', 'compare_strategy']

Intelligence Analyst
 Purpose: Inspect unstructured evidence, distinguish source reliability, identify rumors, and surface information that may confirm or challenge the structured case.
 Tools: ['get_company_snapshot', 'search_documents']

Deal Lead
 Purpose: Integrate specialist memoranda, reconcile disagreement, distinguish facts from judgment, and produce the final M&A recommendation.
 Tools: []

Shared specialist output contract:
{
  "preferred_candidate": "Company ID or NONE

### Code Unit 4 of 10 — Expose role-specific tool schemas instead of one universal toolbox

The next unit translates role boundaries into executable permissions. We create OpenAI function schemas for the deterministic tools and then assemble a different toolbox for each specialist. This matters because a role should be enforced by architecture where possible, not merely requested in natural language. The Financial Analyst literally cannot call the document-search tool because that tool is not exposed to it. The Intelligence Analyst cannot rank candidates using the financial-comparison function because it is not in that agent’s tool list. This creates a clean experiment: when different specialists produce different conclusions, we know that their evidence and responsibilities were intentionally distinct. We also create a shared local registry that maps function names to Python implementations. The result is an important multi-agent pattern: **common infrastructure, differentiated access**. Many real systems use the same idea through permissions, capabilities, or scoped credentials. Here we keep it simple enough to inspect in one cell.

In [5]:
TOOL_SCHEMAS = {
    "get_company_snapshot": {
        "type": "function",
        "name": "get_company_snapshot",
        "description": "Retrieve structured financial and strategic data for one synthetic company.",
        "parameters": {
            "type": "object",
            "properties": {"company_id": {"type": "string"}},
            "required": ["company_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    "screen_candidates": {
        "type": "function",
        "name": "screen_candidates",
        "description": "Filter and rank possible M&A targets using explicit financial criteria.",
        "parameters": {
            "type": "object",
            "properties": {
                "exclude_company_id": {"type": "string"},
                "sector": {"type": "string"},
                "continent": {"type": "string"},
                "min_growth_pct": {"type": "number"},
                "max_ev_ebitda": {"type": "number"},
                "max_net_debt_ebitda": {"type": "number"},
                "limit": {"type": "integer", "minimum": 2, "maximum": 10},
            },
            "required": [
                "exclude_company_id", "sector", "continent", "min_growth_pct",
                "max_ev_ebitda", "max_net_debt_ebitda", "limit"
            ],
            "additionalProperties": False,
        },
        "strict": True,
    },
    "compare_financials": {
        "type": "function",
        "name": "compare_financials",
        "description": "Compare two or more candidates on financial variables.",
        "parameters": {
            "type": "object",
            "properties": {
                "company_ids": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 2,
                    "maxItems": 6,
                }
            },
            "required": ["company_ids"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    "compare_strategy": {
        "type": "function",
        "name": "compare_strategy",
        "description": "Compare target companies on strategic fit relative to a buyer.",
        "parameters": {
            "type": "object",
            "properties": {
                "buyer_id": {"type": "string"},
                "target_ids": {
                    "type": "array",
                    "items": {"type": "string"},
                    "minItems": 2,
                    "maxItems": 6,
                },
            },
            "required": ["buyer_id", "target_ids"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    "search_documents": {
        "type": "function",
        "name": "search_documents",
        "description": "Search financial-report excerpts, analyst notes, and M&A rumors.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "company_id": {"type": "string"},
                "document_type": {
                    "type": "string",
                    "enum": ["ANY", "financial_report_excerpt", "analyst_note", "rumor"],
                },
                "limit": {"type": "integer", "minimum": 1, "maximum": 10},
            },
            "required": ["query", "company_id", "document_type", "limit"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

TOOL_REGISTRY = {
    "get_company_snapshot": get_company_snapshot,
    "screen_candidates": screen_candidates,
    "compare_financials": compare_financials,
    "compare_strategy": compare_strategy,
    "search_documents": search_documents,
}

ROLE_TOOLS = {
    role: [TOOL_SCHEMAS[name] for name in cfg["allowed_tools"]]
    for role, cfg in ROLE_CONFIG.items()
}

for role, tools in ROLE_TOOLS.items():
    print(role, "→", [t["name"] for t in tools])

financial_analyst → ['get_company_snapshot', 'screen_candidates', 'compare_financials']
strategy_analyst → ['get_company_snapshot', 'compare_strategy']
intelligence_analyst → ['get_company_snapshot', 'search_documents']
deal_lead → []


### Code Unit 5 of 10 — Create a reusable specialist-agent runtime

Now that roles and permissions exist, we need a common way to execute a specialist. This cell creates a small agent runtime similar to Notebook 1’s observe–decide–act loop, but parameterized by role. The runtime receives the shared mission plus role-specific instructions, exposes only the tools assigned to that specialist, records an operational audit trace, and returns the specialist’s memorandum. Importantly, the function does not encode what the Financial Analyst or Strategy Analyst should conclude. It provides the mechanism through which each specialist can inspect its permitted evidence and decide what it needs. The output instruction requires a compact JSON object matching the common specialist contract. This makes the specialist’s work easy to pass to another agent later. The cell therefore introduces a core systems idea: many agents can share one execution engine while differing in identity, permissions, objectives, and outputs. The constellation is beginning to emerge from configuration rather than duplicated code.

In [7]:
def specialist_instructions(role_key: str) -> str:
    cfg = ROLE_CONFIG[role_key]
    return f"""
You are the {cfg['title']} in a pedagogical M&A agent constellation.
You operate ONLY on the synthetic TOPIC 766 dataset.

Your responsibility:
{cfg['purpose']}

Rules:
- Stay inside your assigned role.
- Use only the tools you have been given.
- Never invent company facts.
- Do not access or mention any teacher benchmark.
- Distinguish facts from judgment.
- If evidence is insufficient, say so explicitly.

Return the final memorandum as VALID JSON with exactly these keys:
preferred_candidate
alternatives
key_evidence
main_risks
uncertainties
specialist_conclusion
"""


def run_specialist(role_key: str, mission: str, context: str = "", max_rounds: int = 6) -> dict:
    audit = []
    tools = ROLE_TOOLS[role_key]

    full_input = f"""
COMMON MISSION:
{mission}

SHARED CONTEXT:
{context if context else 'No additional shared context.'}
"""

    response = client.responses.create(
        model=MODEL,
        instructions=specialist_instructions(role_key),
        input=full_input,
        tools=tools,
        parallel_tool_calls=False,
    )

    for round_number in range(1, max_rounds + 1):
        calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]

        if not calls:
            try:
                memo = json.loads(response.output_text)
            except Exception:
                memo = {
                    "preferred_candidate": "PARSE_ERROR",
                    "alternatives": [],
                    "key_evidence": [response.output_text],
                    "main_risks": [],
                    "uncertainties": ["Model output was not valid JSON."],
                    "specialist_conclusion": "See raw evidence in key_evidence.",
                }
            return {
                "role": role_key,
                "title": ROLE_CONFIG[role_key]["title"],
                "status": "completed",
                "memo": memo,
                "audit_log": audit,
            }

        outputs = []
        for call in calls:
            args = json.loads(call.arguments)
            result = TOOL_REGISTRY[call.name](**args) if call.name in TOOL_REGISTRY else {
                "ok": False,
                "error": f"Unregistered tool: {call.name}"
            }

            audit.append({
                "round": round_number,
                "tool": call.name,
                "arguments": args,
                "result": result,
            })

            outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result, ensure_ascii=False),
            })

        response = client.responses.create(
            model=MODEL,
            instructions=specialist_instructions(role_key),
            previous_response_id=response.id,
            input=outputs,
            tools=tools,
            parallel_tool_calls=False,
        )

    return {
        "role": role_key,
        "title": ROLE_CONFIG[role_key]["title"],
        "status": "max_rounds_reached",
        "memo": {
            "preferred_candidate": "INCOMPLETE",
            "alternatives": [],
            "key_evidence": [response.output_text],
            "main_risks": [],
            "uncertainties": ["Maximum specialist rounds reached."],
            "specialist_conclusion": "Analysis incomplete.",
        },
        "audit_log": audit,
    }

print("Reusable specialist runtime is ready.")

Reusable specialist runtime is ready.


### Code Unit 6 of 10 — Give the Financial Analyst the first responsibility

The first specialist now performs a concrete piece of the mandate. Rather than letting every agent independently search all 500 companies, we ask the Financial Analyst to identify a financially credible shortlist for buyer `C001`. This is a natural responsibility because candidate generation depends heavily on structured variables such as growth, valuation, and leverage. The analyst can inspect the buyer, choose screening criteria, and compare the strongest candidates financially. Its memorandum becomes the first collaborative artifact in the constellation. Notice the asymmetry we are intentionally creating: the Strategy and Intelligence specialists will not start from the entire universe. They will receive the Financial Analyst’s shortlist as shared context. That is already a form of coordination, because one specialist’s output constrains the subsequent search space. The design is simple and hierarchical, but pedagogically powerful. The learner can now observe how work is divided and how one agent’s artifact becomes another agent’s input.

In [8]:
MISSION = """
Advise synthetic company C001 on a possible acquisition.

The constellation must identify one preferred acquisition target and one credible
alternative. The final recommendation should consider financial attractiveness,
strategic fit, unstructured evidence, source reliability, risks, and uncertainty.
"""

financial_result = run_specialist(
    role_key="financial_analyst",
    mission=MISSION,
    context=(
        "You are the first specialist. Build a financially credible shortlist. "
        "Consider at least three serious candidates if the data permit."
    ),
)

print("FINANCIAL ANALYST MEMORANDUM")
print(json.dumps(financial_result["memo"], indent=2))

financial_shortlist = []
preferred = financial_result["memo"].get("preferred_candidate")
if preferred and preferred not in ["NONE", "PARSE_ERROR", "INCOMPLETE"]:
    financial_shortlist.append(preferred)

for cid in financial_result["memo"].get("alternatives", []):
    if cid not in financial_shortlist:
        financial_shortlist.append(cid)

financial_shortlist = financial_shortlist[:5]

print("\nShortlist passed forward:", financial_shortlist)
print("Tool calls made:", len(financial_result["audit_log"]))

FINANCIAL ANALYST MEMORANDUM
{
  "preferred_candidate": {
    "company_id": "C271",
    "company_name": "Terra Capital 271",
    "rationale": "Best valuation and profitability combination in the available South America Technology set while staying within a leverage band broadly compatible with C001\u2019s already-elevated leverage. Compared with other screened names, C271 offers high EBITDA margin (cash generation) at a lower EV/EBITDA multiple, with solid growth."
  },
  "alternatives": [
    {
      "company_id": "C171",
      "company_name": "Terra Systems 171",
      "rationale": "Credible lower-balance-sheet-risk alternative due to net cash (negative net debt/EBITDA). Growth is comparable to C271, but profitability is materially lower (margin), and valuation is not as attractive as C271."
    }
  ],
  "key_evidence": [
    {
      "acquirer_baseline_C001": {
        "sector_subsector": "Technology / Cybersecurity",
        "revenue_growth_pct": 16.35,
        "ebitda_margin_pct": 

### Code Unit 7 of 10 — Run Strategy and Intelligence specialists over a shared shortlist

The constellation now branches. The Strategy Analyst and Intelligence Analyst receive the same common mission and the shortlist produced by the Financial Analyst, but they examine it through different lenses. The Strategy Analyst compares business models, geography, sector fit, strengths, weaknesses, integration complexity, and regulatory sensitivity. The Intelligence Analyst searches the synthetic document corpus for evidence that may support or challenge the emerging case, with special attention to source reliability. They are not asked to agree. In fact, disagreement is useful because it reveals information that a single generalized analysis might blur together. The key pedagogical idea is that collaboration does not require simultaneous conversation. It can occur through **shared artifacts and sequential handoffs**. The shortlist is one artifact; each specialist memorandum is another. At the end of this cell, we will have three distinct perspectives on the same M&A mandate. This is the moment when multiple agents begin to resemble a team rather than repeated executions of one prompt.

In [11]:
if len(financial_shortlist) < 2:
    buyer = get_company_snapshot("C001")
    buyer_sector = buyer["company"]["sector"]
    fallback = screen_candidates(
        exclude_company_id="C001",
        sector=buyer_sector,
        continent="ANY",
        min_growth_pct=0.0,
        max_ev_ebitda=30.0,
        max_net_debt_ebitda=6.0,
        limit=4,
    )
    financial_shortlist = [x["company_id"] for x in fallback["candidates"]]

SHORTLIST_CONTEXT = f"""
The Financial Analyst produced this candidate shortlist:
{financial_shortlist}

Evaluate these candidates from your assigned specialist perspective.
Do not simply repeat the financial ranking.
"""

strategy_result = run_specialist(
    role_key="strategy_analyst",
    mission=MISSION,
    context=SHORTLIST_CONTEXT,
)

intelligence_result = run_specialist(
    role_key="intelligence_analyst",
    mission=MISSION,
    context=SHORTLIST_CONTEXT,
)

print("STRATEGY ANALYST MEMORANDUM")
print(json.dumps(strategy_result["memo"], indent=2))

print("\nINTELLIGENCE ANALYST MEMORANDUM")
print(json.dumps(intelligence_result["memo"], indent=2))

print("\nTool calls:")
print(" Strategy:", len(strategy_result["audit_log"]))
print(" Intelligence:", len(intelligence_result["audit_log"]))

STRATEGY ANALYST MEMORANDUM
{
  "preferred_candidate": {
    "company_id": "C271",
    "company_name": "Terra Capital 271",
    "specialist_rationale": "Best strategic adjacency to C001 within the provided shortlist: both are Technology in South America, and a cybersecurity buyer can plausibly benefit from semiconductor exposure (hardware security, secure-by-design components, supply-chain assurance). While integration complexity is flagged High and the target is Domestic (Argentina), the operational complementarity (C001 multi-region + C271 domestic) offers a clear geographic expansion path. The business-model mismatch (C001 distribution-led vs C271 project-based) is manageable but requires deliberate operating-model integration."
  },
  "alternatives": [
    {
      "company_id": "C171",
      "company_name": "Terra Systems 171",
      "specialist_rationale": "Credible alternative if C001 prioritizes lower balance-sheet stress and smoother integration: C171 is Multi-region (closer to

### Code Unit 8 of 10 — Build the collaboration packet and expose disagreement

We now have three specialist memoranda, but a pile of outputs is not yet collaboration. This cell turns the separate analyses into a common **collaboration packet**. The packet contains the mission, the shortlist, each specialist’s preferred candidate, evidence, risks, and uncertainties. It also computes a simple agreement summary showing whether the specialists independently converged on the same target or produced conflicting preferences. This is deliberately deterministic. We do not ask another language model to tell us whether two IDs match. The purpose is to separate coordination mechanics from interpretation. If all three specialists favor the same company, the Deal Lead should still inspect their reasons. If they disagree, the disagreement becomes an explicit object that needs reconciliation. This is the first place in the tutorial where an emergent systems issue appears: specialization produces richer perspectives, but those perspectives must be combined. A constellation therefore requires not only agents, but also a communication format and a way to identify conflicts between agents.

In [14]:
specialist_results = {
    "financial_analyst": financial_result,
    "strategy_analyst": strategy_result,
    "intelligence_analyst": intelligence_result,
}

preferences = {
    role: result["memo"].get("preferred_candidate", "NONE")
    for role, result in specialist_results.items()
}

valid_preferences = [
    p for p in preferences.values()
    if p not in ["NONE", "PARSE_ERROR", "INCOMPLETE", None, ""]
]

agreement = {
    "preferences": preferences,
    "unique_preferred_candidates": sorted(set(p["company_id"] for p in valid_preferences if isinstance(p, dict))),
    "full_agreement": len(set(p["company_id"] for p in valid_preferences if isinstance(p, dict))) == 1 and len(valid_preferences) == 3,
    "requires_reconciliation": len(set(p["company_id"] for p in valid_preferences if isinstance(p, dict))) > 1,
}

COLLABORATION_PACKET = {
    "mission": MISSION,
    "buyer_id": "C001",
    "shortlist": financial_shortlist,
    "agreement_summary": agreement,
    "specialist_memoranda": {
        role: result["memo"]
        for role, result in specialist_results.items()
    },
}

print("AGREEMENT SUMMARY")
print(json.dumps(agreement, indent=2))

print("\nCOLLABORATION PACKET READY")
print("Specialists:", list(COLLABORATION_PACKET["specialist_memoranda"].keys()))

AGREEMENT SUMMARY
{
  "preferences": {
    "financial_analyst": {
      "company_id": "C271",
      "company_name": "Terra Capital 271",
      "rationale": "Best valuation and profitability combination in the available South America Technology set while staying within a leverage band broadly compatible with C001\u2019s already-elevated leverage. Compared with other screened names, C271 offers high EBITDA margin (cash generation) at a lower EV/EBITDA multiple, with solid growth."
    },
    "strategy_analyst": {
      "company_id": "C271",
      "company_name": "Terra Capital 271",
      "specialist_rationale": "Best strategic adjacency to C001 within the provided shortlist: both are Technology in South America, and a cybersecurity buyer can plausibly benefit from semiconductor exposure (hardware security, secure-by-design components, supply-chain assurance). While integration complexity is flagged High and the target is Domestic (Argentina), the operational complementarity (C001 multi-

### Code Unit 9 of 10 — Add the Deal Lead as the integrating authority

The Deal Lead is not merely a fourth specialist. It performs a different systems function: integration. Its input is the collaboration packet rather than the raw 500-company dataset. This creates a hierarchy of cognition. Specialists transform raw evidence into focused memoranda; the Deal Lead transforms those memoranda into a decision. The prompt explicitly instructs the lead to identify agreement and disagreement, weigh evidence according to each specialist’s domain, avoid treating low-reliability rumors as facts, and explain why one perspective should dominate when specialists conflict. This is a simple example of **authority design** in a multi-agent system. Someone—or some mechanism—must own the final decision. We deliberately keep the Deal Lead tool-free so it cannot silently redo every specialist’s work. Its job is to synthesize the shared artifacts it receives. The resulting architecture is therefore genuinely modular: evidence gathering and domain analysis occur below; reconciliation and final judgment occur above.

In [15]:
DEAL_LEAD_INSTRUCTIONS = """
You are the Deal Lead of a pedagogical M&A agent constellation.

You receive specialist memoranda from:
1. Financial Analyst
2. Strategy Analyst
3. Intelligence Analyst

Your job is to integrate them, not redo their analyses.

Rules:
- Use only the collaboration packet provided.
- Identify where specialists agree and disagree.
- Give financial facts, strategic judgment, and textual intelligence appropriate weight.
- Treat low-reliability rumors as uncertain signals, never as established facts.
- Do not invent company facts.
- Distinguish evidence from judgment.
- If specialists disagree, explain how you resolved the disagreement.

Return VALID JSON with exactly these keys:
preferred_target
alternative_target
integrated_rationale
specialist_agreement
specialist_disagreement
material_risks
uncertainties
final_recommendation
"""


def run_deal_lead(packet: dict) -> dict:
    response = client.responses.create(
        model=MODEL,
        instructions=DEAL_LEAD_INSTRUCTIONS,
        input=json.dumps(packet, ensure_ascii=False, indent=2),
    )

    try:
        memo = json.loads(response.output_text)
    except Exception:
        memo = {
            "preferred_target": "PARSE_ERROR",
            "alternative_target": "PARSE_ERROR",
            "integrated_rationale": response.output_text,
            "specialist_agreement": [],
            "specialist_disagreement": [],
            "material_risks": [],
            "uncertainties": ["Deal Lead output was not valid JSON."],
            "final_recommendation": "See integrated_rationale.",
        }

    return {"status": "completed", "memo": memo}


deal_lead_result = run_deal_lead(COLLABORATION_PACKET)

print("DEAL LEAD MEMORANDUM")
print(json.dumps(deal_lead_result["memo"], indent=2))

DEAL LEAD MEMORANDUM
{
  "preferred_target": {
    "company_id": "C271",
    "company_name": "Terra Capital 271"
  },
  "alternative_target": {
    "company_id": "C171",
    "company_name": "Terra Systems 171"
  },
  "integrated_rationale": {
    "why_C271_is_preferred": {
      "financial_attractiveness_evidence": [
        "C271 combines high profitability (EBITDA margin 35.52%) with the most attractive valuation in the screened set (EV/EBITDA 12.79) while keeping leverage in a band not obviously worse than C001\u2019s constraint (net debt/EBITDA 1.73 vs C001 1.91).",
        "Growth is solid and comparable to the alternative (C271 revenue growth 14.43%)."
      ],
      "strategic_fit_judgment": [
        "Best adjacency within the shortlist: C001 (Cybersecurity) acquiring a semiconductor platform can plausibly enable differentiated \u201chardware-rooted\u201d security and supply-chain assurance (directional synergy; not validated by product-level data in the packet).",
        "Geo

### Code Unit 10 of 10 — Run and audit the complete constellation as one system

The final unit packages the architecture into a single constellation-level view. We already executed each component step by step for pedagogical transparency; now we assemble their outputs into one system record containing the mission, specialist memoranda, deal-lead recommendation, and operational audit trace. The audit table shows which role used which tool and how many operations each specialist performed. This allows the learner to inspect the division of labor rather than merely reading the final answer. We also calculate a simple role-to-preference table so that agreement and disagreement remain visible. The result should feel materially different from Notebook 1. The answer is no longer produced by one generalist interacting with several tools. It emerges from several bounded analyses that are integrated by an explicit authority. Yet the cell also exposes the next limitation: the constellation executes once. It does not observe what happens after its recommendation, evaluate whether the result was good, or adapt its behavior. That limitation will motivate Notebook 3 and the introduction of loops.

In [16]:
constellation_audit_rows = []

for role, result in specialist_results.items():
    for event in result["audit_log"]:
        constellation_audit_rows.append({
            "role": ROLE_CONFIG[role]["title"],
            "round": event["round"],
            "tool": event["tool"],
            "arguments": json.dumps(event["arguments"], ensure_ascii=False),
        })

CONSTELLATION_RESULT = {
    "mission": MISSION,
    "buyer_id": "C001",
    "shortlist": financial_shortlist,
    "specialists": {
        role: result["memo"]
        for role, result in specialist_results.items()
    },
    "agreement_summary": agreement,
    "deal_lead": deal_lead_result["memo"],
    "audit_trace": constellation_audit_rows,
}

print("=" * 90)
print("FINAL CONSTELLATION RECOMMENDATION")
print("=" * 90)
print(json.dumps(CONSTELLATION_RESULT["deal_lead"], indent=2))

print("\nSPECIALIST PREFERENCES")
display(pd.DataFrame([
    {
        "role": ROLE_CONFIG[role]["title"],
        "preferred_candidate": result["memo"].get("preferred_candidate", "NONE"),
        "conclusion": result["memo"].get("specialist_conclusion", ""),
    }
    for role, result in specialist_results.items()
]))

print("\nOPERATIONAL AUDIT TRACE")
display(pd.DataFrame(constellation_audit_rows))

print("\nPEDAGOGICAL CHECK")
print("NB01: one agent + several tools.")
print("NB02: several specialized agents + role boundaries + collaboration + synthesis.")
print("Remaining limitation: this constellation executes once and does not adapt to outcomes.")
print("Next: NB03 — LOOPS: Observe → Decide → Act → Evaluate → Adapt.")

FINAL CONSTELLATION RECOMMENDATION
{
  "preferred_target": {
    "company_id": "C271",
    "company_name": "Terra Capital 271"
  },
  "alternative_target": {
    "company_id": "C171",
    "company_name": "Terra Systems 171"
  },
  "integrated_rationale": {
    "why_C271_is_preferred": {
      "financial_attractiveness_evidence": [
        "C271 combines high profitability (EBITDA margin 35.52%) with the most attractive valuation in the screened set (EV/EBITDA 12.79) while keeping leverage in a band not obviously worse than C001\u2019s constraint (net debt/EBITDA 1.73 vs C001 1.91).",
        "Growth is solid and comparable to the alternative (C271 revenue growth 14.43%)."
      ],
      "strategic_fit_judgment": [
        "Best adjacency within the shortlist: C001 (Cybersecurity) acquiring a semiconductor platform can plausibly enable differentiated \u201chardware-rooted\u201d security and supply-chain assurance (directional synergy; not validated by product-level data in the packet)."

,role,preferred_candidate,conclusion
0,Financial Analyst,"{'company_id': 'C271', 'company_name': 'Terra ...",On a purely financial screen within South Amer...
1,Strategy Analyst,"{'company_id': 'C271', 'company_name': 'Terra ...","From a strategy and integration perspective, C..."
2,Intelligence Analyst,"{'company_id': 'C271', 'company_name': 'Terra ...",From an intelligence/unstructured-evidence len...



OPERATIONAL AUDIT TRACE


,role,round,tool,arguments
0,Financial Analyst,1,get_company_snapshot,"{""company_id"": ""C001""}"
1,Financial Analyst,2,screen_candidates,"{""exclude_company_id"": ""C001"", ""sector"": ""Tech..."
2,Financial Analyst,3,screen_candidates,"{""exclude_company_id"": ""C001"", ""sector"": ""Tech..."
3,Financial Analyst,4,screen_candidates,"{""exclude_company_id"": ""C001"", ""sector"": ""Tech..."
4,Financial Analyst,5,compare_financials,"{""company_ids"": [""C001"", ""C221"", ""C271"", ""C171..."
5,Strategy Analyst,1,get_company_snapshot,"{""company_id"": ""C001""}"
6,Strategy Analyst,2,get_company_snapshot,"{""company_id"": ""C271""}"
7,Strategy Analyst,3,get_company_snapshot,"{""company_id"": ""C171""}"
8,Intelligence Analyst,1,search_documents,"{""query"": ""C271 Terra Capital 271 acquisition ..."
9,Intelligence Analyst,2,search_documents,"{""query"": ""C171 Terra Systems 171 acquisition ..."



PEDAGOGICAL CHECK
NB01: one agent + several tools.
NB02: several specialized agents + role boundaries + collaboration + synthesis.
Remaining limitation: this constellation executes once and does not adapt to outcomes.
Next: NB03 — LOOPS: Observe → Decide → Act → Evaluate → Adapt.


<div style="padding:20px;border-left:6px solid #16a34a;background:#f0fdf4;border-radius:10px">
<b>Conclusion — From individual agency to organized cognition</b>
</div>

Notebook 2 has introduced a different kind of capability from Notebook 1. We did not primarily add more data, more sophisticated financial formulas, or a more powerful language model. We changed the **organization of cognition**.

In Notebook 1, one agent owned the entire M&A mandate. It could retrieve a buyer, screen companies, compare targets, search documents, and synthesize a recommendation. The architecture was compact and useful, but every analytical perspective existed inside one generalist role. Notebook 2 decomposed that responsibility into a constellation of specialized agents: a Financial Analyst, a Strategy Analyst, an Intelligence Analyst, and a Deal Lead.

The first lesson is that **specialization is architectural, not cosmetic**. A specialist is not created merely by giving an agent a job title. We defined a purpose, a restricted set of tools, and an output contract for each role. The Financial Analyst could work with structured financial evidence but could not search rumors. The Intelligence Analyst could search unstructured evidence but could not perform the financial ranking. The Strategy Analyst examined fit, geography, strengths, weaknesses, integration complexity, and regulatory sensitivity. These boundaries made the division of labor observable and enforceable.

The second lesson is that multiple agents do not automatically form a system. Three independent analyses are merely three outputs. The constellation emerged only when we introduced **shared artifacts and coordination contracts**. The Financial Analyst created a shortlist that constrained later work. The Strategy and Intelligence specialists received that shortlist as shared context. Each specialist returned a memorandum with the same structure. Those memoranda were assembled into a collaboration packet, and the Deal Lead received that packet as the basis for synthesis.

This leads to the third lesson: **collaboration creates a new problem—reconciliation**. Specialization can improve focus, but specialized perspectives may disagree. A financially attractive target may look strategically difficult. A strategically appealing target may carry a rumor that increases uncertainty. Instead of hiding these contradictions inside one long answer, the constellation makes them explicit. The agreement summary tells us whether the specialists converged; the Deal Lead must explain how disagreement was resolved.

The fourth lesson concerns **authority**. A multi-agent system needs some rule for converting partial analyses into action. In this notebook, the authority structure was hierarchical and predetermined. Specialists analyzed; the Deal Lead decided. This is not the only possible design, but it is transparent and easy to teach. Later systems could use voting, negotiation, confidence weighting, markets, or dynamic leadership. For now, a single integrating role gives the learner a clear mental model.

The fifth lesson is that auditability becomes richer in a constellation. We can inspect not only which tools were called, but which **role** called them. This lets us ask whether work was appropriately distributed. Did the Financial Analyst dominate the process? Did the Intelligence Analyst retrieve enough evidence? Did the Strategy Analyst contribute information that materially changed the final recommendation? These questions did not exist in the same form when there was only one agent.

The sixth lesson is that specialization introduces **dependency**. The Strategy and Intelligence specialists received a shortlist produced by the Financial Analyst. This improves efficiency, but it also creates a possible failure path: if the initial shortlist is poor, later specialists may never inspect the best target. The architecture therefore gains focus while also becoming more sensitive to handoff quality. This trade-off will become important when we introduce evaluation and adaptation.

Yet the architecture still has an important limitation. The constellation executes a mission and stops. It does not observe what happens after the recommendation. It does not discover that a rumor became confirmed, that market conditions changed, that a target became too expensive, or that its own recommendation performed poorly. It cannot evaluate the consequences of its action and revise its behavior.

That is the conceptual opening for Notebook 3.

The progression now becomes:

**NB01 — Individual Agency**  
Agent + Tools + Skill

↓

**NB02 — Organized Agency**  
Specialized Agents + Roles + Collaboration

↓

**NB03 — Adaptive Agency**  
Observe + Decide + Act + Evaluate + Adapt

Notebook 3 will therefore ask a fundamentally new question:

> **How can a constellation learn from what happens while it is operating rather than merely executing once?**

We will introduce feedback, state, evaluation, changing evidence, and repeated action. At that point, our M&A system will stop behaving like a sophisticated one-shot workflow and begin to behave like an adaptive process.